# 01. 텍스트 전사문 임베딩 실험

**2026-10-21 / hyungjin**

전체 흐름은 **전사문 → 임베딩 모델 → 임베딩 → 경계 후보 탐색 → LLM 판정**입니다. 이 노트북은 텍스트 입력을 읽고 **임베딩을 생성·검증·저장**하는 부분까지 다룹니다.

`../test/input/*.txt`만 읽습니다. `meetings/*.json`, 정답 주제, 수동 경계는 입력에 사용하지 않습니다.

| 선택값 | 기본 모델 | 호출 방식 |
|---|---|---|
| `huggingface` | `BAAI/bge-m3` | 로컬 `SentenceTransformer.encode()` |
| `openai` | `text-embedding-3-small` | `OpenAI().embeddings.create()` |
| `gemini` | `gemini-embedding-001` | `genai.Client().models.embed_content()` |

기본 실행은 **한 회의의 앞 50개 발화 → 500자 윈도우·100자 오버랩 + Hugging Face**입니다. 작은 입력으로 확인한 뒤 `MAX_BLOCKS = None`으로 전체 회의를 실행하세요. 모델 비교는 `PROVIDER`를 바꾸고 설정 셀부터 다시 실행합니다. 실행마다 별도 폴더에 저장하므로 이전 결과를 덮어쓰지 않습니다.

## 1. 실행 환경 준비

Python 3.12 Jupyter 커널을 선택하세요. 아래 설치 명령은 **주석을 해제하여 필요한 것만** 한 번 실행합니다. 설치 후 import 오류가 남으면 커널을 재시작하세요.

공통 패키지와 선택한 공급자의 패키지만 있으면 됩니다. 로컬 GPU 사용에는 환경에 맞는 PyTorch 설치가 필요하며, GPU가 없으면 CPU로 실행됩니다. BGE-M3 최초 실행은 모델 다운로드와 로딩 시간이 필요합니다.

## 2. 실험 설정과 입력 파일 선택

커널 작업 폴더가 노트북 폴더, `remeet`, 또는 그 상위 프로젝트 폴더여도 경로를 찾습니다. 다른 위치에서 실행한다면 노트북 폴더로 이동하세요.

`INPUT_PATH`를 목록에 나온 다른 `.txt`로 바꿀 수 있습니다. `TRANSCRIPT_TEXT`에 문자열을 넣으면 파일 대신 **그 문자열 자체**를 사용합니다. `None`이면 파일을 읽습니다.

현재 전사문은 `회의 대주제 / 참여자 / 일시` 3줄 머리말 뒤에 빈 줄로 구분한 발화가 이어집니다. 머리말은 별도 보관하고 본문만 임베딩합니다. 머리말이 없는 일반 텍스트는 `HAS_HEADER = False`로 설정하세요. 빈 줄은 원문 발화 위치를 기록하는 데 사용합니다. 실제 임베딩 윈도우는 빈 줄이나 발화 수와 관계없이 본문의 문자 길이로 만듭니다.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
from importlib.metadata import version, PackageNotFoundError
import hashlib
import json
import os
import re
import time
import uuid

import numpy as np
import pandas as pd
from IPython.display import display


def find_experiment_dir():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / "playground/hyungjin/embedding-test",
                          base / "remeet/playground/hyungjin/embedding-test"):
            if (candidate / "01_text_embeddings.ipynb").is_file() and (candidate.parent / "test/input").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("embedding-test 폴더 또는 remeet 폴더에서 커널을 실행하세요.")


EXPERIMENT_DIR = find_experiment_dir()
INPUT_DIR = EXPERIMENT_DIR.parent / "test/input"
INPUT_PATH = INPUT_DIR / "ES2004a.txt"
TRANSCRIPT_TEXT = None  # 직접 입력 예: "첫 번째 발화입니다.\n\n두 번째 발화입니다."
HAS_HEADER = True
MAX_BLOCKS = None  # None: 전체 본문
PROVIDER = "huggingface"  # "huggingface", "openai", "gemini"
MODEL_NAMES = {
    "huggingface": "BAAI/bge-m3",
    "openai": "text-embedding-3-small",  # text-embedding-3-large 등으로 변경 가능
    "gemini": "gemini-embedding-001",
}
MODEL_NAME = MODEL_NAMES[PROVIDER]
BATCH_SIZE = 8
DEVICE = None  # Hugging Face: 자동 선택, 또는 "cpu" / "cuda"
GEMINI_TASK_TYPE = "SEMANTIC_SIMILARITY"
OUTPUT_ROOT = EXPERIMENT_DIR / "outputs"

if BATCH_SIZE < 1 or (MAX_BLOCKS is not None and MAX_BLOCKS < 1):
    raise ValueError("BATCH_SIZE와 MAX_BLOCKS는 양수여야 합니다.")
print("입력 폴더:", INPUT_DIR)
print("사용 가능한 회의:", [p.name for p in sorted(INPUT_DIR.glob("*.txt"))])
print("선택:", PROVIDER, MODEL_NAME)

## 3. 텍스트 길이 기준 슬라이딩 윈도우·오버랩 청킹

**윈도우 크기는 발화 개수가 아닌 입력 텍스트의 길이로 결정합니다.** “네.”, “아니오.” 같은 짧은 응답은 내용 그대로 포함되지만 윈도우 하나를 차지하지 않습니다. 원문의 연속된 텍스트를 같은 길이로 잘라 각 모델에 직접 입력합니다.

모델마다 토크나이저가 다르므로 이번 실험의 공통 청킹 단위는 **문자 수(`len(text)`)**입니다. 공백·줄바꿈·문장부호도 포함하며, 바이트 수나 모델 토큰 수는 아닙니다. 모델별 토큰 한도 검사는 임베딩 단계에서 별도로 적용됩니다.

| 설정 | 의미 | 기본값 |
|---|---|---|
| `WINDOW_CHARS` | 윈도우 하나의 최대 입력 문자 수 | 500 |
| `OVERLAP_CHARS` | 인접 윈도우가 공유할 문자 수 | 100 |
| 이동 간격 | `WINDOW_CHARS - OVERLAP_CHARS` | 400 |

예를 들어 본문이 1,200자면 `[0:500]`, `[400:900]`, `[800:1200]`의 3개 윈도우가 만들어집니다. 이 예시의 위치는 **본문 시작 기준**이며, 결과의 `start_char:end_char`는 머리말을 포함한 `source_text` 기준입니다. 끝 위치는 포함하지 않습니다. 마지막 윈도우를 제외하면 실제 입력은 정확히 500자이고 인접 윈도우는 정확히 100자를 공유합니다.

`WINDOW_CHARS >= 1`, `0 <= OVERLAP_CHARS < WINDOW_CHARS`인 정수만 허용합니다. 오버랩 0은 겹침 없음입니다. 본문 끝을 포함하면 멈추고, 마지막 남은 텍스트는 짧은 윈도우로 보존합니다. 본문이 500자보다 짧으면 윈도우 하나만 만듭니다. 너무 작은 크기로 공백만 있는 윈도우가 생기면 크기를 늘리라는 오류를 표시합니다.

고정 길이를 지키기 위해 **발화·문장·단어 중간에서도 잘릴 수 있습니다.** 경계를 발화 끝으로 옮기거나 짧은 응답을 삭제하지 않습니다. `blocks_df`는 원래 발화/문단 위치를 기록하는 참고 데이터이며 청크 크기를 정하는 데 사용하지 않습니다. `MAX_BLOCKS`는 빠른 실행을 위한 본문 선택 제한으로만 사용하고, 전체 회의는 `None`으로 실행하세요.

`windows_df`의 `char_count`가 실제 임베딩 입력 길이입니다. `start_char:end_char`로 **BOM 제거·LF 줄바꿈 통일 후의 `source_text`**에서 동일 텍스트를 다시 꺼낼 수 있습니다. 모든 모델이 같은 윈도우를 사용하며, 개별 발화 벡터를 평균하지 않습니다.

In [ ]:
WINDOW_CHARS = 500  # 윈도우당 최대 입력 문자 수 (공백·줄바꿈 포함)
OVERLAP_CHARS = 100  # 인접 윈도우가 공유할 문자 수


def parse_transcript(text, has_header=True):
    source = text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")
    spans = []
    cursor = 0
    for separator in re.finditer(r"\n[ \t]*\n(?:[ \t]*\n)*", source):
        spans.append((cursor, separator.start()))
        cursor = separator.end()
    spans.append((cursor, len(source)))
    spans = [(start, end) for start, end in spans if source[start:end].strip()]
    header = ""
    if has_header:
        if not spans:
            raise ValueError("머리말과 본문이 없습니다.")
        start, end = spans.pop(0)
        header = source[start:end].strip()
        if not all(any(line.startswith(prefix) for line in header.splitlines())
                   for prefix in ("회의 대주제:", "참여자:", "일시:")):
            raise ValueError("예상한 머리말이 아닙니다. 일반 텍스트는 HAS_HEADER=False로 설정하세요.")
    rows = []
    for start, end in spans:
        start += len(source[start:end]) - len(source[start:end].lstrip())
        end = start + len(source[start:end].rstrip())
        rows.append({"position": len(rows), "start_char": start, "end_char": end,
                     "text": source[start:end]})
    if not rows:
        raise ValueError("임베딩할 본문이 없습니다.")
    return source, header, rows


def build_windows(blocks, source, window_chars, overlap_chars):
    if (type(window_chars) is not int or type(overlap_chars) is not int
            or window_chars < 1 or not 0 <= overlap_chars < window_chars):
        raise ValueError("윈도우 문자 수는 양의 정수, 오버랩은 0 이상 윈도우 문자 수 미만의 정수여야 합니다.")
    if not blocks:
        raise ValueError("윈도우를 만들 본문이 없습니다.")
    body_start, body_end = blocks[0]["start_char"], blocks[-1]["end_char"]
    windows = []
    for start_char in range(body_start, body_end, window_chars - overlap_chars):
        end_char = min(start_char + window_chars, body_end)
        text = source[start_char:end_char]
        if not text.strip():
            raise ValueError("공백만 포함한 윈도우가 있습니다. WINDOW_CHARS를 늘리세요.")
        windows.append({"window_index": len(windows), "start_char": start_char,
                        "end_char": end_char, "char_count": len(text), "text": text})
        if end_char == body_end:
            break
    return windows


if TRANSCRIPT_TEXT is None:
    raw_bytes = INPUT_PATH.read_bytes()
    input_text = raw_bytes.decode("utf-8-sig")
    input_label = str(INPUT_PATH.resolve())
else:
    input_text = TRANSCRIPT_TEXT
    raw_bytes = input_text.encode("utf-8")
    input_label = "inline_text"
source_sha256 = hashlib.sha256(raw_bytes).hexdigest()
source_text, meeting_header, all_blocks = parse_transcript(input_text, HAS_HEADER)
blocks = all_blocks if MAX_BLOCKS is None else all_blocks[:MAX_BLOCKS]
blocks_df = pd.DataFrame(blocks)
windows = build_windows(blocks, source_text, WINDOW_CHARS, OVERLAP_CHARS)
windows_df = pd.DataFrame(windows)
texts = windows_df["text"].tolist()
assert all(source_text[r["start_char"]:r["end_char"]] == r["text"] for r in blocks)
print(meeting_header or "머리말 없는 텍스트")
print(f"전체 {len(all_blocks)}블록 중 {len(blocks)}블록 사용")
print(f"윈도우 {WINDOW_CHARS}자 / 오버랩 {OVERLAP_CHARS}자 / 이동 간격 {WINDOW_CHARS - OVERLAP_CHARS}자")
print(f"임베딩할 윈도우: {len(windows)}개")
display(blocks_df.head(10))
display(windows_df.head(10))
display(windows_df["char_count"].describe().rename("윈도우 입력 문자 수 (토큰 수와 다름)"))
print("첫 윈도우의 실제 입력 텍스트:\n", texts[0])

## 4. API 키 설정 (로컬 모델이면 건너뛰기)

OpenAI는 `OPENAI_API_KEY`, Gemini는 `GEMINI_API_KEY` 또는 `GOOGLE_API_KEY`를 환경변수로 설정합니다. 프로젝트 상위에서 `.env`를 찾으면 읽되 기존 환경변수를 덮어쓰지 않습니다. 키가 없으면 실행 시 숨김 입력창을 사용합니다. 키를 코드나 출력에 적지 않습니다.

**OpenAI / Gemini를 선택하고 생성 셀을 실행하면 본문이 해당 외부 API로 전송되며 사용량에 따라 비용이 발생할 수 있습니다.** Hugging Face는 모델 다운로드 후 로컬에서 본문을 처리합니다.

In [ ]:
def get_api_key(provider):
    from getpass import getpass
    from dotenv import load_dotenv

    for parent in (EXPERIMENT_DIR, *EXPERIMENT_DIR.parents):
        env_path = parent / ".env"
        if env_path.is_file():
            load_dotenv(env_path, override=False)
            break
    names = ("OPENAI_API_KEY",) if provider == "openai" else ("GEMINI_API_KEY", "GOOGLE_API_KEY")
    key = next((os.environ[name] for name in names if os.environ.get(name)), "")
    if not key:
        key = getpass(f"{names[0]}: ").strip()
    if not key:
        raise ValueError(f"{names[0]} is required.")
    return key

## 5. 공급자별 임베딩 생성 함수

반환 형태는 모두 `(윈도우 수, 임베딩 차원)` 행렬과 실행 정보입니다. 선택한 공급자만 import하므로 세 SDK를 모두 설치할 필요는 없습니다.

- **Hugging Face:** Sentence Transformers 호환 모델을 사용합니다. 특수 토큰을 포함한 길이를 확인하고 모델 한도를 넘으면 자르지 않고 중단합니다. BGE-M3는 본문을 그대로 넣습니다. 다른 모델로 바꿀 때는 모델 카드의 prefix/prompt 요구사항도 확인하세요.
- **OpenAI:** `tiktoken`으로 입력별 최대 8,192토큰, 요청당 합계 300,000토큰, 최대 2,048개 입력을 넘지 않게 배치합니다. 응답 `index`로 입력 순서를 확인하고 사용 토큰도 기록합니다.
- **Gemini:** `google-genai`와 `gemini-embedding-001`을 사용합니다. 문자열 목록의 각 문자열마다 벡터를 받습니다. 주제 변화 비교를 위한 대칭적 유사도 실험이므로 `SEMANTIC_SIMILARITY`를 사용합니다. 모델별 설정이므로 세 공급자의 처리 방식이 완전히 같다는 뜻은 아닙니다.

Gemini는 여기서 정확한 사전 토큰 계수를 제공하지 않습니다. **짧은 윈도우 실험에 한정해 1,800 UTF-8 바이트를 넘는 윈도우을 거부**합니다. 이는 공식 토큰 한도가 아닌 보수적인 실험 제한입니다. 긴 입력을 다루려면 정확한 토큰 계수·초과 처리 방식을 확인한 뒤 확장하세요. API 한도 오류가 나도 자동 분할하거나 다른 모델로 대체하지 않습니다.

`gemini-embedding-2`는 여러 입력의 집계 방식과 task 설정이 달라 이름만 바꾸는 것을 지원하지 않습니다. 다른 Gemini 모델은 호출 방식을 별도로 확인해야 합니다.

In [ ]:
def embed_texts(texts, provider, model_name, batch_size=8, device=None,
                gemini_task_type="SEMANTIC_SIMILARITY"):
    if not texts or any(not isinstance(text, str) or not text.strip() for text in texts):
        raise ValueError("입력은 비어 있지 않은 문자열 목록이어야 합니다.")
    if batch_size < 1:
        raise ValueError("batch_size는 양수여야 합니다.")
    info = {"provider": provider, "model": model_name, "batch_size": batch_size}
    started = time.perf_counter()
    if provider == "huggingface":
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer(model_name, device=device)
        info["model_load_seconds"] = time.perf_counter() - started
        limit = int(model.max_seq_length)
        counts = []
        for start in range(0, len(texts), batch_size):
            tokenized = model.tokenizer(texts[start:start + batch_size],
                                        truncation=False, add_special_tokens=True)
            counts.extend(len(ids) for ids in tokenized["input_ids"])
        too_long = [i for i, count in enumerate(counts) if count > limit]
        if too_long:
            raise ValueError(f"모델 한도 {limit}토큰 초과 입력: {too_long[:10]}; 먼저 분할하세요.")
        vectors = model.encode(texts, batch_size=batch_size, show_progress_bar=True,
                               convert_to_numpy=True, normalize_embeddings=False)
        info.update(device=str(model.device), input_token_limit=limit,
                    input_token_counts=counts,
                    hf_revision=getattr(model[0].auto_model.config, "_commit_hash", None))
    elif provider == "openai":
        import tiktoken
        from openai import OpenAI

        encoding = tiktoken.encoding_for_model(model_name)
        counts = [len(encoding.encode(text, disallowed_special=())) for text in texts]
        if any(count > 8192 for count in counts):
            raise ValueError("OpenAI 입력별 8192토큰 한도 초과: 먼저 분할하세요.")
        vectors = []
        prompt_tokens = 0
        response_models = set()
        with OpenAI(api_key=get_api_key(provider), timeout=60.0, max_retries=2) as client:
            start = 0
            while start < len(texts):
                end, request_tokens = start, 0
                while (end < len(texts) and end - start < min(batch_size, 2048)
                       and request_tokens + counts[end] <= 300000):
                    request_tokens += counts[end]
                    end += 1
                response = client.embeddings.create(model=model_name, input=texts[start:end],
                                                    encoding_format="float")
                items = sorted(response.data, key=lambda item: item.index)
                if [item.index for item in items] != list(range(end - start)):
                    raise ValueError("OpenAI 응답의 입력 인덱스가 일치하지 않습니다.")
                vectors.extend(item.embedding for item in items)
                prompt_tokens += response.usage.prompt_tokens
                response_models.add(response.model)
                print(f"OpenAI: {end}/{len(texts)}윈도우 완료")
                start = end
        info.update(input_token_counts=counts, input_token_limit=8192,
                    api_prompt_tokens=prompt_tokens, response_models=sorted(response_models))
    elif provider == "gemini":
        from google import genai
        from google.genai import types

        if model_name != "gemini-embedding-001":
            raise ValueError("이 Gemini 경로는 gemini-embedding-001 전용입니다.")
        byte_counts = [len(text.encode("utf-8")) for text in texts]
        # shortcut: Gemini는 짧은 입력만 허용, 긴 입력 실험 시 정확한 토큰 계수와 초과 처리 추가.
        if any(count > 1800 for count in byte_counts):
            raise ValueError("Gemini 실험 제한인 입력당 1800 UTF-8 바이트 초과: 먼저 분할하세요.")
        vectors = []
        with genai.Client(api_key=get_api_key(provider),
                          http_options=types.HttpOptions(timeout=60000)) as client:
            for start in range(0, len(texts), min(batch_size, 100)):
                batch = texts[start:start + min(batch_size, 100)]
                response = client.models.embed_content(
                    model=model_name, contents=batch,
                    config=types.EmbedContentConfig(task_type=gemini_task_type))
                if response.embeddings is None or len(response.embeddings) != len(batch):
                    raise ValueError("Gemini 응답 벡터 수와 입력 입력 수가 다릅니다.")
                vectors.extend(item.values for item in response.embeddings)
                print(f"Gemini: {min(start + len(batch), len(texts))}/{len(texts)}윈도우 완료")
        info.update(task_type=gemini_task_type, input_token_counts=None,
                    input_byte_counts=byte_counts, experiment_max_input_bytes=1800)
    else:
        raise ValueError(f"지원하지 않는 공급자: {provider}")
    info["total_seconds"] = time.perf_counter() - started
    return np.asarray(vectors, dtype=np.float32), info


def normalize_embeddings(vectors, expected_rows):
    vectors = np.asarray(vectors, dtype=np.float32)
    if vectors.ndim != 2 or vectors.shape[0] != expected_rows or vectors.shape[1] == 0:
        raise ValueError("임베딩의 행 수 또는 차원이 올바르지 않습니다.")
    if not np.isfinite(vectors).all():
        raise ValueError("임베딩에 NaN/Inf가 있습니다.")
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    if not np.isfinite(norms).all() or (norms <= 1e-12).any():
        raise ValueError("임베딩에 정규화할 수 없는 벡터가 있습니다.")
    return vectors / norms

## 6. 선택한 모델 실행 및 결과 확인

이 셀부터 실제 모델 로딩 또는 API 호출이 발생합니다. CPU 실행이 느리면 `MAX_BLOCKS`나 `BATCH_SIZE`를 줄이세요. API 인증·권한·할당량 오류가 나면 키와 계정 사용량을 확인하세요.

모든 벡터를 **L2 정규화**하여 길이를 1로 맞춥니다. 같은 모델의 두 벡터 내적은 코사인 유사도로 사용할 수 있습니다. 모델마다 차원과 벡터 공간이 다르므로 **다른 모델의 벡터끼리 직접 내적하면 안 됩니다.** 같은 경계 탐색 방법의 결과를 비교해야 합니다.

In [ ]:
raw_embeddings, embedding_info = embed_texts(
    texts, PROVIDER, MODEL_NAME, BATCH_SIZE, DEVICE, GEMINI_TASK_TYPE)
embeddings = normalize_embeddings(raw_embeddings, len(texts))
assert embeddings.dtype == np.float32
assert np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-5)
print(f"shape={embeddings.shape} (윈도우 수, 벡터 차원)")
print(f"실측 소요 시간: {embedding_info['total_seconds']:.2f}초 (로딩/통신 포함)")
print("메모리:", embeddings.nbytes, "bytes")
print("첫 벡터의 앞 8개 값:", embeddings[0, :8])
print("사용 토큰 (API 보고값):", embedding_info.get("api_prompt_tokens", "미제공"))

## 7. 다음 노트북을 위한 결과 저장

`outputs/<실행시각>-<공급자>-<고유값>/`에 다음 파일을 저장합니다.

| 파일 | 내용 |
|---|---|
| `embeddings.npz` | `embeddings`: 정규화한 float32 행렬 |
| `blocks.jsonl` | 원본 기본 블록의 순서·문자 위치·텍스트 (벡터 행과는 별도) |
| `windows.jsonl` | 벡터 각 행의 윈도우 순서·문자 위치·실제 입력 문자 수·텍스트 |
| `source.txt` | 문자 위치의 기준인 정규화한 전체 입력 텍스트 |
| `manifest.json` | 입력 해시, 모델·패키지 버전, 문자 단위 윈도우 크기·오버랩·이동 간격, 행 수·차원, 실측 시간 |

원본 입력 해시는 원본 파일 바이트(직접 입력은 UTF-8 바이트), `normalized_source_sha256`은 저장한 `source.txt` 기준입니다. 부분 실행 여부도 기록합니다. 출력은 전사문을 포함하므로 이 폴더의 `.gitignore`로 Git에서 제외합니다. API 키는 저장하지 않습니다.

모델/API 결과는 제공자 업데이트에 따라 달라질 수 있습니다. Hugging Face는 확보 가능한 모델 커밋을 기록하지만 외부 API의 내부 가중치 버전은 고정하지 못합니다.

In [ ]:
package_versions = {}
for package in ("numpy", "pandas", "sentence-transformers", "torch", "openai", "tiktoken", "google-genai"):
    try:
        package_versions[package] = version(package)
    except PackageNotFoundError:
        pass
run_id = (datetime.now(timezone(timedelta(hours=9))).strftime("%Y%m%d-%H%M%S")
          + f"-{PROVIDER}-" + uuid.uuid4().hex[:8])
output_dir = OUTPUT_ROOT / run_id
output_dir.mkdir(parents=True, exist_ok=False)
manifest = {
    "schema_version": 3, "run_id": run_id, "status": "embeddings_only",
    "input": input_label, "source_sha256": source_sha256,
    "normalized_source_sha256": hashlib.sha256(source_text.encode("utf-8")).hexdigest(),
    "header": meeting_header, "has_header": HAS_HEADER,
    "block_parse_rule": "blank_line_blocks; strip_outer_whitespace; preserve_order",
    "split_rule": "fixed_length_characters; fixed_stride; preserve_source_text",
    "position_convention": "zero_based; source.txt character offsets; end exclusive",
    "text_input": "direct embedding of overlapping body windows sliced from source.txt",
    "window_chars": WINDOW_CHARS, "overlap_chars": OVERLAP_CHARS,
    "stride_chars": WINDOW_CHARS - OVERLAP_CHARS, "window_unit": "unicode_character",
    "length_rule": "Python len(text); includes whitespace and LF newlines",
    "tail_policy": "keep_short_final_window; stop_when_end_is_covered",
    "max_blocks": MAX_BLOCKS, "total_block_count": len(all_blocks),
    "selected_block_count": len(blocks), "window_count": len(windows), "is_partial": len(blocks) < len(all_blocks),
    "embedding_shape": list(embeddings.shape), "dtype": str(embeddings.dtype),
    "normalization": "L2 per row", "embedding_info": embedding_info,
    "package_versions": package_versions,
}
np.savez_compressed(output_dir / "embeddings.npz", embeddings=embeddings)
with (output_dir / "blocks.jsonl").open("w", encoding="utf-8", newline="\n") as stream:
    for block in blocks:
        stream.write(json.dumps(block, ensure_ascii=False) + "\n")
with (output_dir / "windows.jsonl").open("w", encoding="utf-8", newline="\n") as stream:
    for window in windows:
        stream.write(json.dumps(window, ensure_ascii=False) + "\n")
with (output_dir / "source.txt").open("w", encoding="utf-8", newline="\n") as stream:
    stream.write(source_text)
(output_dir / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장 완료:", output_dir)

## 8. 저장 결과만 읽어 재사용하기

이 셀은 API나 모델을 다시 호출하지 않습니다. 다른 커널에서는 `saved_dir`에 결과 폴더를 지정하고 `Path`, `json`, `hashlib`, `numpy`를 import하세요. 행 수·차원·원문 해시·각 윈도우의 실제 길이·이동 간격·오버랩을 검사합니다.

다음 경계 후보 탐색 노트북에는 `loaded_windows`와 `loaded_embeddings`를 넘깁니다. **임베딩 행 `i`는 `loaded_windows[i]`의 텍스트 벡터**입니다. `loaded_blocks`는 원본 발화/문단 위치를 보관하는 참고 목록이며 임베딩 행과 1:1 대응하지 않습니다.

`start_char:end_char`로 후보 주변 원문을 찾아 LLM 입력을 만들 수 있습니다. 청크 시작과 끝은 발화 중간일 수도 있습니다. 원래 발화 위치가 필요하면 `loaded_blocks`의 문자 범위와 대조하세요. 인접 윈도우에 공유 텍스트가 있으므로 윈도우 사이를 곧바로 확정 경계로 해석하지 않습니다. 여기서는 후보 점수나 LLM 판정을 계산하지 않습니다.

문자 길이 방식의 출력은 `schema_version=3`입니다. 이전 발화 개수 방식의 결과와 혼용하지 말고 새로 생성한 결과를 지정하세요.

In [ ]:
saved_dir = output_dir  # 다른 결과를 읽으려면 Path("outputs/실행폴더")로 변경
loaded_manifest = json.loads((saved_dir / "manifest.json").read_text(encoding="utf-8"))
assert loaded_manifest["schema_version"] == 3, "문자 길이 방식으로 다시 생성한 결과가 필요합니다."
loaded_blocks = [json.loads(line) for line in
                 (saved_dir / "blocks.jsonl").read_text(encoding="utf-8").splitlines()]
loaded_windows = [json.loads(line) for line in
                  (saved_dir / "windows.jsonl").read_text(encoding="utf-8").splitlines()]
with np.load(saved_dir / "embeddings.npz", allow_pickle=False) as archive:
    loaded_embeddings = archive["embeddings"]
loaded_source = (saved_dir / "source.txt").read_text(encoding="utf-8")
assert hashlib.sha256(loaded_source.encode("utf-8")).hexdigest() == loaded_manifest["normalized_source_sha256"]
assert list(loaded_embeddings.shape) == loaded_manifest["embedding_shape"]
assert len(loaded_windows) == loaded_embeddings.shape[0] == loaded_manifest["window_count"]
assert len(loaded_blocks) == loaded_manifest["selected_block_count"]
assert [w["window_index"] for w in loaded_windows] == list(range(len(loaded_windows)))
assert loaded_manifest["window_unit"] == "unicode_character"
assert loaded_windows[0]["start_char"] == loaded_blocks[0]["start_char"]
assert loaded_windows[-1]["end_char"] == loaded_blocks[-1]["end_char"]
for i, window in enumerate(loaded_windows):
    start, end = window["start_char"], window["end_char"]
    assert window["text"] == loaded_source[start:end]
    assert window["char_count"] == len(window["text"]) == end - start
    assert 0 < window["char_count"] <= loaded_manifest["window_chars"]
    if i < len(loaded_windows) - 1:
        assert window["char_count"] == loaded_manifest["window_chars"]
        following = loaded_windows[i + 1]
        assert following["start_char"] - start == loaded_manifest["stride_chars"]
        assert end - following["start_char"] == loaded_manifest["overlap_chars"]
assert [row["position"] for row in loaded_blocks] == list(range(len(loaded_blocks)))
assert all(loaded_source[row["start_char"]:row["end_char"]] == row["text"] for row in loaded_blocks)
assert np.isfinite(loaded_embeddings).all()
assert np.allclose(np.linalg.norm(loaded_embeddings, axis=1), 1.0, atol=1e-5)
print("재로딩 검증 완료:", loaded_embeddings.shape)

## 공식 문서와 실험 해석

- [BGE-M3 모델 카드](https://huggingface.co/BAAI/bge-m3): 모델 사용법과 특성.
- [Sentence Transformers](https://sbert.net/docs/package_reference/sentence_transformer/model.html): 로딩·인코딩.
- [OpenAI 가이드](https://developers.openai.com/api/docs/guides/embeddings), [Python API](https://developers.openai.com/api/reference/python/resources/embeddings/methods/create): 호출과 입력 한도.
- [Gemini 가이드](https://ai.google.dev/gemini-api/docs/embeddings), [001 모델 사양](https://ai.google.dev/gemini-api/docs/models/gemini-embedding-001): task 설정과 모델 사용법.

현재 확인하는 것은 입력·출력 정합성, 차원, 실행 시간입니다. 경계 탐지 품질은 아직 측정하지 않았습니다. 다음 단계에서 같은 텍스트 범위·같은 문자 길이 윈도우와 오버랩·같은 경계 탐색 설정으로 비교하고, 이후 LLM에 후보 주변 원문을 보내 판정합니다. 기본값은 일부 블록만 처리하므로 전체 회의 실험 전에는 `MAX_BLOCKS = None`으로 다시 생성하세요.